# Session 10: Exercises

**Before session 11.** Predict, then measure. Answers go in the **✍️** cells, and code TODOs are
marked `# TODO`.

Most of these call the real model through the graphs, so each exercise says roughly how many calls
it makes. Exercises 2 and 4 make **none**. Every policy search also embeds its question with
`gemini-embedding-2`, so you need `GEMINI_API_KEY` in `labs/.env`, and the knowledge base must exist
(`uv run python kb/ingest.py` from `project/`).

Several exercises change the imported v7 modules **in memory** (a desk added, a router swapped). The
snapshot on disk stays v7, and each exercise puts things back when it ends.

## Setup: given, just run it

In [ ]:
import operator
import sys
import threading
import time
import warnings
from pathlib import Path
from typing import Annotated, Literal, TypedDict

warnings.filterwarnings("ignore", message="IProgress not found")    # fastembed's progress bar

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))

from langchain_core.callbacks import BaseCallbackHandler      # noqa: E402
from langchain_core.documents import Document                 # noqa: E402
from langgraph.graph import END, START, StateGraph            # noqa: E402
from langgraph.types import Command, Send                     # noqa: E402
from pydantic import BaseModel, Field                         # noqa: E402

from config import settings                                   # noqa: E402
from evals import run_eval                                    # noqa: E402
from inbox import answer_ticket, load_all_tickets             # noqa: E402
from kb import fusion                                         # noqa: E402
from middleware import Customer, customer_for                 # noqa: E402
from utils import ask_json, build_model                       # noqa: E402
from workflows.inbox import graph as wiring                   # noqa: E402
from workflows.inbox import nodes                             # noqa: E402
from workflows.inbox.state import InboxState                  # noqa: E402

model = build_model()
TICKETS = load_all_tickets()


class CountModelCalls(BaseCallbackHandler):
    """Counts every chat-model call in a run, including calls inside a desk's
    agent and inside parallel branches (hence the lock)."""
    def __init__(self):
        self.calls = 0
        self._lock = threading.Lock()

    def on_chat_model_start(self, *args, **kwargs):
        with self._lock:
            self.calls += 1


def mermaid(graph) -> str:
    return graph.get_graph().draw_mermaid().split("---")[-1].strip()


print(f"v{settings.VERSION} · {settings.model_id} · {len(TICKETS)} tickets")

## ⭐ Exercise 1: A product desk

**What this teaches:** adding a worker to an orchestrator-worker graph touches the router's table and
the wiring. No node that already exists has to change.

`product_question` lands on the `general` desk, which holds six tools. Give it its own desk instead,
holding only `get_product`. In the project this is a few lines in `workflows/inbox/nodes.py` and
`workflows/inbox/graph.py`. Here you make the same additions to the imported modules.

1. Add a `"product"` entry to `nodes.DESKS` (a one-line remit, and the one tool).
2. Point `product_question` at it in `nodes.DESK_FOR`.
3. Write `product_desk(task, runtime)` and add it to `wiring.DESK_NODES`.
4. Build the graph, print its diagram, and run TCK-M04 (about 12 model calls). The kettle question
   should reach the product desk.

In [ ]:
from tools import get_product    # noqa: E402

# TODO 1-3: the desk's entry, the route, the node function and its registration


inbox = wiring.build_inbox()
print(mermaid(inbox))
final = answer_ticket(inbox, TICKETS["TCK-M04"], steps=True)
print("\n" + final["reply"])

**✍️ Your answer.** Which existing node did you have to change, and why is that the point of the pattern?

In [ ]:
# Put v7 back for the next exercises (run this even if your exercise 1 is unfinished).
nodes.DESKS.pop("product", None)
wiring.DESK_NODES.pop("product", None)
nodes.DESK_FOR["product_question"] = "general"

## ⭐⭐ Exercise 2: Reducer bug hunt

**What this teaches:** which reducer to choose, and why the one that sounds right can be wrong.

This graph fans out to two nodes that both write `notes`, and it crashes. No model is involved.

In [ ]:
class Audit(TypedDict):
    ticket_id: str
    notes: list[str]


def check_identity(state: Audit) -> dict:
    return {"notes": ["identity: sender matches the order's customer"]}


def check_window(state: Audit) -> dict:
    return {"notes": ["window: delivered 21 days ago"]}


def build_audit(state_type):
    builder = StateGraph(state_type)
    builder.add_node("identity", check_identity)
    builder.add_node("window", check_window)
    builder.add_edge(START, "identity")
    builder.add_edge(START, "window")
    builder.add_edge("identity", END)
    builder.add_edge("window", END)
    return builder.compile()


try:
    build_audit(Audit).invoke({"ticket_id": "TCK-002", "notes": []})
except Exception as err:
    print(type(err).__name__, "-", str(err).splitlines()[0])

1. Fix it with the right reducer and show both notes survive.
2. Someone suggests `add_messages`, since "it's also for lists". Try it, and print the type of each
   item in `notes`.

In [ ]:
# TODO 1: a fixed state type, and the run

# TODO 2: the add_messages version, and the type of each note

**✍️ Your answer.** Why is `add_messages` wrong for this key, even though it doesn't crash?

## ⭐⭐ Exercise 3: The cheap router, proven

**What this teaches:** the price of a routing decision is measurable, and "an LLM router is smarter"
is a claim you can test.

v7 routes each sub-issue with a dict lookup. The alternative many tutorials show is an LLM router:
ask the model which desk should take it.

1. Run TCK-M01 through v7 with `CountModelCalls` attached (`config={"callbacks": [counter]}`) and
   print the number of model calls.
2. Build a variant whose router asks the model, once per sub-issue, using `DeskChoice` below. Count
   again.
3. Compare the two counts and the desks each chose.

About 25 model calls in all.

In [ ]:
class DeskChoice(BaseModel):
    desk: Literal["orders", "billing", "policy", "general"] = Field(
        description="orders: where an order is or when it ships. billing: charges and refunds. "
                    "policy: returns and warranty rules. general: anything else.")

In [ ]:
m01 = TICKETS["TCK-M01"]
ayesha = customer_for(m01["from"])

# TODO 1: count v7's model calls on TCK-M01

# TODO 2: llm_route(state) -> list[Send], and a graph that uses it
#         (copy wiring.build_inbox's wiring, swapping the router)

# TODO 3: count again, and print each version's desks

**✍️ Your answer.** What did the LLM router buy for its extra calls, and when would it be worth paying for?

## ⭐⭐ Exercise 4: RRF by hand

**What this teaches:** reciprocal-rank fusion is a formula you can run on paper, and it needs no
score normalisation.

Three phrasings of one question returned these ranked lists of passage ids (best first):

| Query | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| q1 | A | B | C | D | E |
| q2 | C | A | F | B | G |
| q3 | F | C | A | H | B |

1. **On paper first**: compute each id's RRF score, `Σ 1/(60 + rank)`, and write down the fused top 4.
2. Check your answer against `kb/fusion.rrf` using the stand-in `Document`s below. No model, no search.
3. Then try `k=1` instead of 60. Does the top 4 change? Why?

In [ ]:
LISTS = {"q1": "ABCDE", "q2": "CAFBG", "q3": "FCAHB"}


def doc(pid: str) -> Document:
    """A stand-in passage: rrf only needs a stable identity, which is metadata["_id"]."""
    return Document(page_content=f"passage {pid}", metadata={"_id": pid, "source": "demo", "section": pid})


ranked = [[doc(p) for p in ids] for ids in LISTS.values()]

In [ ]:
# TODO 1: your paper answer, as a list of 4 ids
mine = []

# TODO 2: fusion.rrf(ranked, k=60, keep=4) — compare with yours

# TODO 3: the same with k=1

**✍️ Your answer.** Why does RRF need no normalisation, when averaging the three queries' cosine
scores would? What does a small `k` do?

## ⭐⭐⭐ Exercise 5: An escalation rail, with `Command`

**What this teaches:** routing from inside the node that did the work, and putting a hand-off to a
person on the graph's own path.

ShopWise's rule: an **angry customer on a high-urgency ticket** never gets an automated answer. A
person writes back, and they start from a hand-off summary.

Build a variant of the inbox graph where:

1. a node `triage_or_escalate` runs triage (`nodes.triage_node`) and returns a `Command`. It goes to
   `escalate` if `urgency == "high"` and `sentiment == "angry"`, and to `split` otherwise;
2. `escalate` writes a three-line hand-off summary for a person (one model call) into `reply`, and ends;
3. there is **no** edge `triage → split`, because the `Command` does the routing.

Run it on TCK-003 (should escalate) and TCK-M02 (should not). About 10 model calls.

In [ ]:
class HandOff(BaseModel):
    what_happened: str = Field(description="One sentence: the customer's problem, with order ids.")
    what_they_want: str = Field(description="One sentence.")
    risk: str = Field(description="One sentence: why this needs a person now.")

In [ ]:
# TODO: must_escalate(triage_result) -> bool

# TODO: triage_or_escalate(state) -> Command[Literal["split", "escalate"]]

# TODO: escalate(state) -> {"reply": ...}

# TODO: build the variant and run TCK-003 and TCK-M02

**✍️ Your answer.** Why `Command` from the triage node rather than a conditional edge after it? Two sentences.

## ⭐⭐⭐ Exercise 6: The retry budget

**What this teaches:** a one-variable staircase at system scale. A retry cap is a cost decision, and
it has a measurable answer.

The corrective loop's ceiling is `settings.RAG_MAX_ATTEMPTS`: searches allowed per question, the first
included. At `1` the grader runs but can never send the question round again. Score the graded policy
graph on session 9's gold questions at `RAG_MAX_ATTEMPTS` = 1, 2 and 3, holding everything else still.
Use `k=2`, so retrieval misses often enough for retries to matter.

Report refusal accuracy, retrieval and calls per question for each step, then pick a value. About 19 ×
(2 to 4) calls per step, roughly 150 in all. Set `LIMIT` lower to spend less.

In [ ]:
LIMIT = None
rows = run_eval.questions()[:LIMIT]
original = settings.RAG_MAX_ATTEMPTS

# TODO: for each budget in (1, 2, 3): set settings.RAG_MAX_ATTEMPTS, score
#       run_eval.graph_target(grade=True, k=2) on rows, keep the result.
#       Restore the original value afterwards (try/finally).

# TODO: print one row per budget: refusal, retrieval, calls/q, and a text bar for calls/q

**✍️ Your answer.** Which budget do you pick, and what is the `RAG-DECISIONS.md` row for it? Where does
the curve flatten?

## [ADV] Exercise 7: The Shop Owner's Daily Brief, part 1

**What this teaches:** the textbook workflow. The steps never vary, so nothing in it should be an
agent.

`report/gather.py` already has the three facts-gathering steps: `sales_by_product_type`,
`open_refunds` and `ticket_volume`. Each is pure Python, and each returns raw data. Build the rest as
a graph:

- three **gather** nodes, run in parallel from `START`, each writing its own key;
- three **writer** nodes, each writing one short paragraph (one model call) from *its own* gather's
  data only;
- an `assemble` node that joins the three paragraphs under a dated heading and writes the markdown to
  `labs/.state/brief-<day>.md`.

Then answer: do the three gather nodes need a reducer? Why is this a workflow, and which section might
one day justify an agent? Three model calls.

In [ ]:
from report import gather    # noqa: E402

DAY = gather.REPORT_DAY
STATE_DIR = settings.QDRANT_PATH.parent          # labs/.state, gitignored

# TODO: Brief state, three gather nodes, three writers, assemble, the wiring, one run

**✍️ Your answer.** Do the gather nodes need a reducer? Why is this a workflow, and what might one day justify an agent?

## Before session 11

Run TCK-M01 with `uv run python inbox.py TCK-M01 --steps` and read the billing desk's note about
REF-9001. Write one sentence: what would the graph need in order to *issue* that refund safely, rather
than promise that a manager will look? Hold onto it. Session 11 builds exactly that.